# ハンズオン(2)改善編 — recipe が効くことを before/after で測る

[診断編](hands_on_diagnosis.ipynb)で見たとおり、`weak_relaxation` の recipe は
「整数×連続の積を `mk.linearize_product` で**厳密線形化**する」こと。
この編ではそれが**定量的に効く**題材で、`mk.compare_variants` を使って
改善前後をルート双対境界・gap・ノード数で比較する。

### 題材の選び方(診断センサスの含意)

[診断センサス](../census.md)が示したのは「現代SCIPは強く、教科書的改善の多くは
自動で解消される」という現実だった(緩いBig-M・対称性・被約コスト固定はSCIPが自動処理)。
その中で **SCIPが自動ではやらない**=モデラーが作り込む価値がある改善の代表が、
**整数×連続の積の厳密線形化**である。

題材は **バッチ反応器スケジューリング**
(`samples/others/scheduling_plant.py`)。バッチ数 `n`(整数)× バッチサイズ `s`(連続)の
積が需要充足の三重積に入り、SCIPは双線形を McCormick 緩和するため緩和が弱くなる。
`build_model(linearize_ns=True)` で `mk.linearize_product` による厳密線形化版に切り替わる。

In [1]:
import sys, pathlib
# リポジトリルート(samples/ を持つ階層)を探して import パスに載せる。
# nbconvert 実行時の cwd は notebook のディレクトリなので、上に辿って解決する。
ROOT = pathlib.Path.cwd()
while not (ROOT / "samples").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/physics_and_control_minlp", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. ベースラインを診断する

まず双線形のまま(`linearize_ns=False`)のモデルを `mk.analyze` に掛け、
診断編と同じ `weak_relaxation` が出ることを確認する。

In [2]:
import minlpkit as mk
import scheduling_plant as sp

report = mk.analyze(lambda: sp.build_model(linearize_ns=False),
                    name="plant-baseline", time_limit=10)
print(report.summary())
print("\n支配ボトルネック:", report.metrics.get("bottleneck_type"),
      "/ 空間分枝の寄与:", f"{report.metrics.get('spatial_share', 0):.1%}")

[plant-baseline] 検出症状 4件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  [good] 制約-変数がブロック構造 + 少数の結合制約 -> ベンダーズ分解 / Dantzig-Wolfe分解(結合制約を主問題に)

支配ボトルネック: energy / 空間分枝の寄与: 54.4%


三重積 `n·s·(T−T0)`(energy 制約)がボトルネックとして出る。
recipe は「整数×連続は `mk.linearize_product`」。次でその中身を見る。

## 2. 改善の中身 — `mk.linearize_product`

整数変数 `n ∈ {1,…,N}` と連続変数 `s ∈ [0, S]` の積 `n·s` は、
`n` を指示変数 `δ_v`(n=v のとき1)で分解すると

```
n = Σ_v v·δ_v ,   Σ_v δ_v = 1 ,   n·s = Σ_v v·s_v
```

と表せる(`s_v` は `δ_v=1` のときだけ `s` に等しい連続変数)。
これは **McCormick のような緩和ではなく厳密**なので、線形緩和のギャップが 0 になる。
`scheduling_plant.build_model(linearize_ns=True)` はこの1行

```python
ns = mk.linearize_product(m, n, s, n_lb, n_ub, s_lb, s_ub, name)
```

に置き換えるだけで三重積を落としている。汎用ヘルパーなので任意モデルに1行で適用できる。

## 3. before/after 比較 — `mk.compare_variants`

双線形のまま(baseline)と厳密線形化(linearized)を同じ時間制限で解き、
**ルート双対境界・最終双対境界・最終gap・ノード数**を1表にまとめる。

> 定式化の質は「ルート双対境界」で測るのが交絡がない。時間制限内のgap/ノードは
> 探索動学に左右されるが、参考として並べる。

In [3]:
df = mk.compare_variants(
    {"baseline(bilinear)": lambda: sp.build_model(linearize_ns=False),
     "linearized":         lambda: sp.build_model(linearize_ns=True)},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,baseline(bilinear),52.128421,88.184431,1.619667,2157
1,linearized,133.302921,140.677591,0.641713,361


In [4]:
base = df.loc[df["variant"] == "baseline(bilinear)"].iloc[0]
lin  = df.loc[df["variant"] == "linearized"].iloc[0]
print(f"ルート双対境界 : {base['root_dual']:.1f} → {lin['root_dual']:.1f} "
      f"(+{(lin['root_dual']/base['root_dual']-1)*100:.0f}%)")
print(f"最終gap        : {base['final_gap']:.1%} → {lin['final_gap']:.1%}")
print(f"ノード数       : {int(base['nodes'])} → {int(lin['nodes'])}")

ルート双対境界 : 52.1 → 133.3 (+156%)
最終gap        : 162.0% → 64.2%
ノード数       : 2157 → 361


## 4. 読み解き

- **ルート双対境界が大きく上がる**(約 52 → 133、+150%規模)。
  McCormick緩和のギャップが消え、最小化問題の下界が最初から締まる。
- 締まった下界のぶん**最終gapもノード数も縮む**。同じ時間で証明が進む。
- 最適値そのものは変わらない(等価変換)。**緩和だけを締める**改善である。

### なぜSCIPが自動でやらないのか

SCIPは双線形項に汎用の McCormick 緩和を当てる。「整数側を指示変数で分解すれば厳密になる」
という**問題構造に依存した再定式化**は、build済みモデルから自動検出できない
(SCIPですら再定式化は自動化しない)。だから診断が構造を指摘し、
モデラーが `mk.linearize_product` を1行当てる、という分業になる。

### 正直な注意 — 効果は問題依存

同じ recipe でも、易しいモデルでは効果は小さい。例えば
[クイックスタート](quickstart.ipynb)のトイ問題や `samples/others/scheduling.py` では
ルート双対境界がほぼ動かない(元々ギャップが小さい)。
効果が大きいのは、このプラントのように**双線形項が本当に律速**になっているモデルである。
だからこそ最初に `mk.analyze` で `weak_relaxation` を確認し、
`mk.compare_variants` で**効いたことを測ってから**採用する、という手順が要になる。